# 12 - El Empujon Final: De 58.8% a 60%+

## Situacion actual
- **GRU_Attention**: 64.3% Val, **58.8% Test**, 91.2% Top-5 (EL CAMPEON)
- Ensemble ingenuo: 55.4% Val, 58.8% Test (no mejora porque mezcla modelos malos)

## Estrategias para el 1.2% que falta

1. **Weighted Ensemble**: Dar 80% del peso al GRU_Attention
2. **TTA Afinado**: TTA solo para el campeon con mas augmentaciones
3. **Self-Ensemble**: 3 versiones del GRU_Attention con diferentes seeds

---

In [1]:
# ==========================================
# 1. IMPORTS Y SETUP
# ==========================================
import numpy as np
import json
from pathlib import Path
from typing import Dict, List, Tuple
import random
from tqdm import tqdm
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
# ==========================================
# 2. RUTAS
# ==========================================
BASE = Path("/home/fransando/Dev/tfg-sign-language-recognition")
EXPERIMENTS = BASE / "05_experiments"
DATASETS = BASE / "06_datasets/wlasl/landmarks"

# Modelos existentes
LSTM_PATH = EXPERIMENTS / "lstm_landmarks/best_model.pt"
GRU_PATH = EXPERIMENTS / "lstm_landmarks_wlasl30_gru_suave/best_model.pt"
GRU_ENRICHED_PATH = EXPERIMENTS / "ensemble_final/best_model_enriched.pt"
GRU_ATTENTION_PATH = EXPERIMENTS / "final_assault/best_model_attention.pt"

# Datasets
WLASL30_ROOT = DATASETS / "wlasl30"
FILTERED_ROOT = DATASETS / "wlasl100_filtered30"

# Output
OUTPUT_DIR = EXPERIMENTS / "final_push_60"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Modelos disponibles:")
print(f"  LSTM: {LSTM_PATH.exists()}")
print(f"  GRU: {GRU_PATH.exists()}")
print(f"  GRU Enriched: {GRU_ENRICHED_PATH.exists()}")
print(f"  GRU Attention: {GRU_ATTENTION_PATH.exists()} <-- EL CAMPEON")

Modelos disponibles:
  LSTM: True
  GRU: True
  GRU Enriched: True
  GRU Attention: True <-- EL CAMPEON


In [3]:
# ==========================================
# 3. CARGAR METADATA
# ==========================================
with open(WLASL30_ROOT / 'metadata.json') as f:
    meta = json.load(f)

gloss_to_idx = meta['gloss_to_idx']
idx_to_gloss = {int(k): v for k, v in meta['idx_to_gloss'].items()}
NUM_CLASSES = len(gloss_to_idx)
print(f"Clases: {NUM_CLASSES}")

Clases: 30


In [4]:
# ==========================================
# 4. DEFINIR ARQUITECTURAS
# ==========================================

class PoseClassifier(nn.Module):
    """Modelo base LSTM/GRU."""
    def __init__(self, config: Dict):
        super().__init__()
        input_size = config['input_size']
        hidden_size = config['hidden_size']
        num_layers = config['num_layers']
        num_classes = config['num_classes']
        dropout = config['dropout']
        bidirectional = config['bidirectional']
        model_type = config.get('model_type', 'LSTM')
        
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        self.input_projection = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        RNN_CLASS = nn.LSTM if model_type == 'LSTM' else nn.GRU
        self.rnn = RNN_CLASS(
            input_size=hidden_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        self.model_type = model_type
        
        rnn_output_size = hidden_size * self.num_directions
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(rnn_output_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )
    
    def forward(self, x):
        x = self.input_projection(x)
        if self.model_type == 'LSTM':
            _, (h_n, _) = self.rnn(x)
        else:
            _, h_n = self.rnn(x)
        if self.bidirectional:
            h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)
        else:
            h_final = h_n[-1]
        return self.classifier(h_final)


class TemporalAttention(nn.Module):
    def __init__(self, hidden_size: int):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.Tanh(),
            nn.Linear(hidden_size // 2, 1, bias=False)
        )
    
    def forward(self, rnn_output):
        scores = self.attention(rnn_output).squeeze(-1)
        weights = F.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), rnn_output).squeeze(1)
        return context, weights


class PoseClassifierAttention(nn.Module):
    """GRU con Attention - EL CAMPEON."""
    def __init__(self, config: Dict):
        super().__init__()
        input_size = config['input_size']
        hidden_size = config['hidden_size']
        num_layers = config['num_layers']
        num_classes = config['num_classes']
        dropout = config['dropout']
        bidirectional = config.get('bidirectional', True)
        
        self.num_directions = 2 if bidirectional else 1
        rnn_hidden = hidden_size * self.num_directions
        
        self.input_projection = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.gru = nn.GRU(
            input_size=hidden_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        
        self.attention = TemporalAttention(rnn_hidden)
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(rnn_hidden, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )
    
    def forward(self, x, return_attention=False):
        x = self.input_projection(x)
        rnn_out, _ = self.gru(x)
        context, attn_weights = self.attention(rnn_out)
        logits = self.classifier(context)
        if return_attention:
            return logits, attn_weights
        return logits


print("Arquitecturas definidas.")

Arquitecturas definidas.


In [5]:
# ==========================================
# 5. DATASET Y TTA TRANSFORMS
# ==========================================

class TTATransform:
    @staticmethod
    def rotate(landmarks, angle_deg):
        landmarks = landmarks.copy()
        angle = np.radians(angle_deg)
        cos_a, sin_a = np.cos(angle), np.sin(angle)
        x, y = landmarks[:, :, 0].copy(), landmarks[:, :, 1].copy()
        landmarks[:, :, 0] = x * cos_a - y * sin_a
        landmarks[:, :, 1] = x * sin_a + y * cos_a
        return landmarks
    
    @staticmethod
    def scale(landmarks, factor):
        return landmarks * factor
    
    @staticmethod
    def flip_horizontal(landmarks):
        landmarks = landmarks.copy()
        landmarks[:, :, 0] = -landmarks[:, :, 0]
        return landmarks
    
    @staticmethod
    def add_noise(landmarks, std=0.005):
        return landmarks + np.random.normal(0, std, landmarks.shape)
    
    @staticmethod
    def translate(landmarks, tx, ty):
        landmarks = landmarks.copy()
        landmarks[:, :, 0] += tx
        landmarks[:, :, 1] += ty
        return landmarks


# TTA extendido (10 transforms)
TTA_TRANSFORMS = [
    lambda x: x,  # Original (peso alto)
    lambda x: TTATransform.rotate(x, 3),
    lambda x: TTATransform.rotate(x, -3),
    lambda x: TTATransform.rotate(x, 6),
    lambda x: TTATransform.rotate(x, -6),
    lambda x: TTATransform.scale(x, 0.97),
    lambda x: TTATransform.scale(x, 1.03),
    lambda x: TTATransform.add_noise(x, 0.003),
    lambda x: TTATransform.translate(x, 0.01, 0),
    lambda x: TTATransform.translate(x, -0.01, 0),
]


class PoseDataset(Dataset):
    def __init__(self, root_dir: Path, split: str, gloss_to_idx: Dict,
                 max_frames: int = 64, augmentation=None):
        self.root_dir = root_dir
        self.max_frames = max_frames
        self.augmentation = augmentation
        self.gloss_to_idx = gloss_to_idx
        
        self.samples = []
        split_dir = root_dir / split
        
        for gloss_dir in split_dir.iterdir():
            if not gloss_dir.is_dir():
                continue
            gloss = gloss_dir.name
            if gloss not in gloss_to_idx:
                continue
            label = gloss_to_idx[gloss]
            for npy_file in gloss_dir.glob('*.npy'):
                self.samples.append((npy_file, label, gloss))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        npy_path, label, _ = self.samples[idx]
        landmarks = np.load(npy_path)
        landmarks = self._pad_or_truncate(landmarks)
        if self.augmentation:
            landmarks = self.augmentation(landmarks)
        landmarks = landmarks.reshape(self.max_frames, -1)
        return torch.tensor(landmarks, dtype=torch.float32), label
    
    def get_raw_landmarks(self, idx):
        """Devuelve landmarks sin procesar para TTA."""
        npy_path, label, _ = self.samples[idx]
        landmarks = np.load(npy_path)
        landmarks = self._pad_or_truncate(landmarks)
        return landmarks, label
    
    def _pad_or_truncate(self, landmarks):
        T = landmarks.shape[0]
        if T > self.max_frames:
            indices = np.linspace(0, T-1, self.max_frames, dtype=int)
            landmarks = landmarks[indices]
        elif T < self.max_frames:
            pad_size = self.max_frames - T
            padding = np.zeros((pad_size, 21, 3))
            landmarks = np.concatenate([landmarks, padding], axis=0)
        return landmarks


# Crear datasets
val_dataset = PoseDataset(WLASL30_ROOT, 'val', gloss_to_idx)
test_dataset = PoseDataset(WLASL30_ROOT, 'test', gloss_to_idx)

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Val: {len(val_dataset)} samples")
print(f"Test: {len(test_dataset)} samples")

Val: 56 samples
Test: 34 samples


In [6]:
# ==========================================
# 6. CARGAR TODOS LOS MODELOS
# ==========================================

models = {}

# LSTM
if LSTM_PATH.exists():
    ckpt = torch.load(LSTM_PATH, map_location=device)
    config = ckpt['config'].copy()
    config['model_type'] = 'LSTM'
    model = PoseClassifier(config).to(device)
    state_dict = ckpt['model_state_dict']
    # Renombrar keys si es necesario
    new_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith('lstm.'):
            new_state_dict['rnn.' + k[5:]] = v
        else:
            new_state_dict[k] = v
    model.load_state_dict(new_state_dict)
    model.eval()
    models['LSTM'] = model

# GRU
if GRU_PATH.exists():
    ckpt = torch.load(GRU_PATH, map_location=device)
    model = PoseClassifier(ckpt['config']).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    models['GRU'] = model

# GRU Enriched
if GRU_ENRICHED_PATH.exists():
    ckpt = torch.load(GRU_ENRICHED_PATH, map_location=device)
    model = PoseClassifier(ckpt['config']).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    models['GRU_Enriched'] = model

# GRU Attention (EL CAMPEON)
if GRU_ATTENTION_PATH.exists():
    ckpt = torch.load(GRU_ATTENTION_PATH, map_location=device)
    model = PoseClassifierAttention(ckpt['config']).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    models['GRU_Attention'] = model
    CONFIG_ATTENTION = ckpt['config']

print(f"\nModelos cargados: {list(models.keys())}")
print(f"CAMPEON: GRU_Attention")


Modelos cargados: ['LSTM', 'GRU', 'GRU_Enriched', 'GRU_Attention']
CAMPEON: GRU_Attention


In [7]:
# ==========================================
# 7. FUNCIONES DE EVALUACION
# ==========================================

@torch.no_grad()
def evaluate(models_list, loader, weights=None, top_k=(1, 5)):
    """Evaluacion con ensemble ponderado."""
    if weights is None:
        weights = [1.0 / len(models_list)] * len(models_list)
    
    for m in models_list:
        m.eval()
    
    total = 0
    top_k_correct = {k: 0 for k in top_k}
    
    for batch, labels in loader:
        batch, labels = batch.to(device), labels.to(device)
        
        ensemble_logits = None
        for model, w in zip(models_list, weights):
            logits = model(batch)
            if ensemble_logits is None:
                ensemble_logits = w * logits
            else:
                ensemble_logits += w * logits
        
        total += labels.size(0)
        _, pred_topk = ensemble_logits.topk(max(top_k), 1, True, True)
        for k in top_k:
            correct_k = pred_topk[:, :k].eq(labels.view(-1, 1).expand_as(pred_topk[:, :k]))
            top_k_correct[k] += correct_k.any(dim=1).sum().item()
    
    return {f'top{k}': 100 * top_k_correct[k] / total for k in top_k}


@torch.no_grad()
def evaluate_single_with_tta(model, dataset, n_augments=10, original_weight=0.4):
    """
    TTA afinado para un solo modelo.
    Da mas peso a la prediccion original.
    """
    model.eval()
    
    total = 0
    correct_top1, correct_top5 = 0, 0
    
    # Peso para cada augmentacion
    aug_weight = (1 - original_weight) / (n_augments - 1) if n_augments > 1 else 0
    
    for idx in tqdm(range(len(dataset)), desc="TTA Afinado"):
        landmarks, label = dataset.get_raw_landmarks(idx)
        
        weighted_logits = None
        
        for i, transform in enumerate(TTA_TRANSFORMS[:n_augments]):
            lm_aug = transform(landmarks.copy())
            lm_aug = lm_aug.reshape(1, 64, -1)
            lm_tensor = torch.tensor(lm_aug, dtype=torch.float32).to(device)
            
            logits = model(lm_tensor)
            
            # Peso: original tiene mas peso
            w = original_weight if i == 0 else aug_weight
            
            if weighted_logits is None:
                weighted_logits = w * logits
            else:
                weighted_logits += w * logits
        
        total += 1
        _, pred_top5 = weighted_logits.topk(5, 1, True, True)
        label_tensor = torch.tensor([label], device=device)
        
        if pred_top5[0, 0] == label:
            correct_top1 += 1
        if label in pred_top5[0]:
            correct_top5 += 1
    
    return {
        'top1': 100 * correct_top1 / total,
        'top5': 100 * correct_top5 / total
    }


print("Funciones de evaluacion definidas.")

Funciones de evaluacion definidas.


---
# BASELINE: Resultados actuales
---

In [8]:
# ==========================================
# 8. BASELINE - RESULTADOS ACTUALES
# ==========================================

print("=" * 60)
print("BASELINE - RESULTADOS INDIVIDUALES")
print("=" * 60)

baseline_results = {}
for name, model in models.items():
    val_m = evaluate([model], val_loader)
    test_m = evaluate([model], test_loader)
    baseline_results[name] = {'val': val_m, 'test': test_m}
    marker = " <-- CAMPEON" if name == 'GRU_Attention' else ""
    print(f"{name:<20} Val: {val_m['top1']:.1f}%  Test: {test_m['top1']:.1f}% (Top-5: {test_m['top5']:.1f}%){marker}")

print("\n--- Ensemble Ingenuo (pesos iguales) ---")
all_models = list(models.values())
naive_val = evaluate(all_models, val_loader)
naive_test = evaluate(all_models, test_loader)
print(f"Ensemble:            Val: {naive_val['top1']:.1f}%  Test: {naive_test['top1']:.1f}% (Top-5: {naive_test['top5']:.1f}%)")

BASELINE - RESULTADOS INDIVIDUALES
LSTM                 Val: 37.5%  Test: 41.2% (Top-5: 67.6%)
GRU                  Val: 53.6%  Test: 38.2% (Top-5: 76.5%)
GRU_Enriched         Val: 51.8%  Test: 47.1% (Top-5: 79.4%)
GRU_Attention        Val: 64.3%  Test: 58.8% (Top-5: 91.2%) <-- CAMPEON

--- Ensemble Ingenuo (pesos iguales) ---
Ensemble:            Val: 51.8%  Test: 58.8% (Top-5: 79.4%)


---
# OPCION 1: Weighted Ensemble (5 minutos)
---

In [9]:
# ==========================================
# 9. OPCION 1: WEIGHTED ENSEMBLE
# ==========================================

print("\n" + "=" * 60)
print("OPCION 1: WEIGHTED ENSEMBLE")
print("=" * 60)
print("Dar 80% del peso al GRU_Attention (el campeon)")

# Orden: LSTM, GRU, GRU_Enriched, GRU_Attention
model_order = ['LSTM', 'GRU', 'GRU_Enriched', 'GRU_Attention']
models_ordered = [models[n] for n in model_order if n in models]

# Probar diferentes configuraciones de pesos
weight_configs = [
    ([0.25, 0.25, 0.25, 0.25], "Igual (baseline)"),
    ([0.1, 0.1, 0.1, 0.7], "70% Attention"),
    ([0.05, 0.05, 0.1, 0.8], "80% Attention"),
    ([0.05, 0.05, 0.05, 0.85], "85% Attention"),
    ([0.0, 0.0, 0.1, 0.9], "90% Attention + 10% Enriched"),
    ([0.0, 0.0, 0.0, 1.0], "100% Attention (solo)"),
]

best_weighted = None
best_test_weighted = 0

print(f"\n{'Config':<30} {'Val Top-1':<12} {'Test Top-1':<12} {'Test Top-5':<12}")
print("-" * 70)

for weights, name in weight_configs:
    # Ajustar pesos si faltan modelos
    w = weights[:len(models_ordered)]
    
    val_m = evaluate(models_ordered, val_loader, w)
    test_m = evaluate(models_ordered, test_loader, w)
    
    print(f"{name:<30} {val_m['top1']:.1f}%{'':<7} {test_m['top1']:.1f}%{'':<7} {test_m['top5']:.1f}%")
    
    if test_m['top1'] > best_test_weighted:
        best_test_weighted = test_m['top1']
        best_weighted = (w, name, val_m, test_m)

print(f"\nMejor config: {best_weighted[1]}")
print(f"  Test Top-1: {best_weighted[3]['top1']:.1f}%")


OPCION 1: WEIGHTED ENSEMBLE
Dar 80% del peso al GRU_Attention (el campeon)

Config                         Val Top-1    Test Top-1   Test Top-5  
----------------------------------------------------------------------
Igual (baseline)               51.8%        58.8%        79.4%
70% Attention                  58.9%        61.8%        85.3%
80% Attention                  60.7%        55.9%        88.2%
85% Attention                  60.7%        55.9%        88.2%
90% Attention + 10% Enriched   64.3%        58.8%        88.2%
100% Attention (solo)          64.3%        58.8%        91.2%

Mejor config: 70% Attention
  Test Top-1: 61.8%


---
# OPCION 2: TTA Afinado para el Campeon
---

In [10]:
# ==========================================
# 10. OPCION 2: TTA AFINADO
# ==========================================

print("\n" + "=" * 60)
print("OPCION 2: TTA AFINADO PARA GRU_ATTENTION")
print("=" * 60)
print("10 augmentaciones, 40% peso a la original")

gru_attention = models['GRU_Attention']

# Probar diferentes configuraciones
tta_configs = [
    (5, 0.3, "5 augs, 30% original"),
    (5, 0.5, "5 augs, 50% original"),
    (10, 0.3, "10 augs, 30% original"),
    (10, 0.4, "10 augs, 40% original"),
    (10, 0.5, "10 augs, 50% original"),
]

best_tta = None
best_test_tta = 0

print(f"\n{'Config':<25} {'Val Top-1':<12} {'Test Top-1':<12} {'Test Top-5':<12}")
print("-" * 65)

for n_aug, orig_w, name in tta_configs:
    val_m = evaluate_single_with_tta(gru_attention, val_dataset, n_aug, orig_w)
    test_m = evaluate_single_with_tta(gru_attention, test_dataset, n_aug, orig_w)
    
    print(f"{name:<25} {val_m['top1']:.1f}%{'':<7} {test_m['top1']:.1f}%{'':<7} {test_m['top5']:.1f}%")
    
    if test_m['top1'] > best_test_tta:
        best_test_tta = test_m['top1']
        best_tta = (n_aug, orig_w, name, val_m, test_m)

print(f"\nMejor TTA: {best_tta[2]}")
print(f"  Test Top-1: {best_tta[4]['top1']:.1f}%")


OPCION 2: TTA AFINADO PARA GRU_ATTENTION
10 augmentaciones, 40% peso a la original

Config                    Val Top-1    Test Top-1   Test Top-5  
-----------------------------------------------------------------


TTA Afinado: 100%|██████████| 34/34 [00:00<00:00, 83.29it/s]


5 augs, 30% original      64.3%        61.8%        91.2%


TTA Afinado: 100%|██████████| 34/34 [00:00<00:00, 71.34it/s]


5 augs, 50% original      64.3%        61.8%        91.2%


...

10 augs, 30% original     64.3%        58.8%        91.2%


TTA Afinado: 100%|██████████| 34/34 [00:00<00:00, 43.85it/s]


10 augs, 40% original     64.3%        58.8%        91.2%


TTA Afinado: 100%|██████████| 34/34 [00:00<00:00, 48.48it/s]

10 augs, 50% original     64.3%        58.8%        91.2%

Mejor TTA: 5 augs, 30% original
  Test Top-1: 61.8%


---
# OPCION 3: Self-Ensemble (3 GRU_Attention)
---

In [11]:
# ==========================================
# 11. AUGMENTATION PARA ENTRENAMIENTO
# ==========================================

class SkeletalAugmentation:
    def __init__(self, rotation=10, scale=(0.9, 1.1), translation=0.1, noise=0.01, p=0.5):
        self.rotation = rotation
        self.scale = scale
        self.translation = translation
        self.noise = noise
        self.p = p
    
    def __call__(self, landmarks):
        landmarks = landmarks.copy()
        if random.random() < self.p:
            angle = np.radians(random.uniform(-self.rotation, self.rotation))
            cos_a, sin_a = np.cos(angle), np.sin(angle)
            x, y = landmarks[:, :, 0].copy(), landmarks[:, :, 1].copy()
            landmarks[:, :, 0] = x * cos_a - y * sin_a
            landmarks[:, :, 1] = x * sin_a + y * cos_a
        if random.random() < self.p:
            landmarks = landmarks * random.uniform(*self.scale)
        if random.random() < self.p:
            landmarks[:, :, 0] += random.uniform(-self.translation, self.translation)
            landmarks[:, :, 1] += random.uniform(-self.translation, self.translation)
        if random.random() < self.p:
            landmarks = landmarks + np.random.normal(0, self.noise, landmarks.shape)
        if random.random() < self.p * 0.5:
            landmarks[:, :, 0] = -landmarks[:, :, 0]
        return landmarks

In [12]:
# ==========================================
# 12. FUNCION PARA ENTRENAR GRU_ATTENTION
# ==========================================

def train_gru_attention(seed: int, config: Dict, train_root: Path, 
                        save_path: Path, epochs: int = 150, patience: int = 30):
    """
    Entrena un GRU_Attention con una seed especifica.
    """
    # Fijar seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
    
    # Crear datasets
    train_aug = SkeletalAugmentation()
    train_ds = PoseDataset(train_root, 'train', gloss_to_idx, augmentation=train_aug)
    val_ds = PoseDataset(train_root, 'val', gloss_to_idx)
    
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
    val_loader_local = DataLoader(val_ds, batch_size=32, shuffle=False)
    
    # Crear modelo
    model = PoseClassifierAttention(config).to(device)
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=5e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    
    best_val = 0
    patience_counter = 0
    
    print(f"\nEntrenando GRU_Attention (seed={seed})...")
    
    for epoch in range(epochs):
        model.train()
        for batch, labels in train_loader:
            batch, labels = batch.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(batch)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        
        scheduler.step()
        
        # Eval
        val_m = evaluate([model], val_loader_local)
        
        if val_m['top1'] > best_val:
            best_val = val_m['top1']
            patience_counter = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'val_top1': best_val,
                'seed': seed,
                'config': config
            }, save_path)
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            break
        
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1}: Val {val_m['top1']:.1f}% (Best: {best_val:.1f}%)")
    
    print(f"  Finalizado. Best Val: {best_val:.1f}%")
    
    # Cargar mejor modelo
    ckpt = torch.load(save_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    
    return model, best_val

In [13]:
# ==========================================
# 13. OPCION 3: SELF-ENSEMBLE
# ==========================================

print("\n" + "=" * 60)
print("OPCION 3: SELF-ENSEMBLE (3x GRU_Attention)")
print("=" * 60)
print("Entrenar 2 modelos adicionales con seeds diferentes")

# Ya tenemos el modelo original (seed 42)
attention_models = [models['GRU_Attention']]
attention_names = ['Seed_42 (original)']

# Usar dataset enriquecido
TRAIN_ROOT = FILTERED_ROOT if FILTERED_ROOT.exists() else WLASL30_ROOT

# Entrenar con seeds adicionales
additional_seeds = [123, 999]

for seed in additional_seeds:
    save_path = OUTPUT_DIR / f'gru_attention_seed{seed}.pt'
    
    if save_path.exists():
        print(f"\nCargando modelo existente (seed={seed})...")
        ckpt = torch.load(save_path, map_location=device)
        model = PoseClassifierAttention(ckpt['config']).to(device)
        model.load_state_dict(ckpt['model_state_dict'])
        model.eval()
        best_val = ckpt['val_top1']
    else:
        model, best_val = train_gru_attention(
            seed=seed,
            config=CONFIG_ATTENTION,
            train_root=TRAIN_ROOT,
            save_path=save_path
        )
    
    attention_models.append(model)
    attention_names.append(f'Seed_{seed}')
    
    # Evaluar individualmente
    test_m = evaluate([model], test_loader)
    print(f"  Seed {seed}: Test Top-1: {test_m['top1']:.1f}%")

print(f"\nModelos de atencion: {len(attention_models)}")


OPCION 3: SELF-ENSEMBLE (3x GRU_Attention)
Entrenar 2 modelos adicionales con seeds diferentes

Entrenando GRU_Attention (seed=123)...
  Epoch 20: Val 51.8% (Best: 51.8%)
  Epoch 40: Val 48.2% (Best: 64.3%)
  Epoch 60: Val 57.1% (Best: 64.3%)
  Finalizado. Best Val: 64.3%
  Seed 123: Test Top-1: 52.9%

Entrenando GRU_Attention (seed=999)...
  Epoch 20: Val 48.2% (Best: 55.4%)
  Epoch 40: Val 53.6% (Best: 60.7%)
  Epoch 60: Val 53.6% (Best: 66.1%)
  Finalizado. Best Val: 66.1%
  Seed 999: Test Top-1: 52.9%

Modelos de atencion: 3


In [14]:
# ==========================================
# 14. EVALUAR SELF-ENSEMBLE
# ==========================================

print("\n--- Self-Ensemble de GRU_Attention ---")

# Individuales
print(f"\n{'Modelo':<25} {'Val Top-1':<12} {'Test Top-1':<12} {'Test Top-5':<12}")
print("-" * 65)

for name, model in zip(attention_names, attention_models):
    val_m = evaluate([model], val_loader)
    test_m = evaluate([model], test_loader)
    print(f"{name:<25} {val_m['top1']:.1f}%{'':<7} {test_m['top1']:.1f}%{'':<7} {test_m['top5']:.1f}%")

# Ensemble de los 3
print("-" * 65)
self_ensemble_val = evaluate(attention_models, val_loader)
self_ensemble_test = evaluate(attention_models, test_loader)
print(f"{'SELF-ENSEMBLE (3x)':<25} {self_ensemble_val['top1']:.1f}%{'':<7} {self_ensemble_test['top1']:.1f}%{'':<7} {self_ensemble_test['top5']:.1f}%")


--- Self-Ensemble de GRU_Attention ---

Modelo                    Val Top-1    Test Top-1   Test Top-5  
-----------------------------------------------------------------
Seed_42 (original)        64.3%        58.8%        91.2%
Seed_123                  64.3%        52.9%        82.4%
Seed_999                  66.1%        52.9%        85.3%
-----------------------------------------------------------------
SELF-ENSEMBLE (3x)        67.9%        55.9%        85.3%


In [15]:
# ==========================================
# 15. SELF-ENSEMBLE + TTA (LA BOMBA FINAL)
# ==========================================

print("\n" + "=" * 60)
print("COMBINACION FINAL: SELF-ENSEMBLE + TTA")
print("=" * 60)

@torch.no_grad()
def evaluate_ensemble_with_tta(models_list, dataset, n_augments=10, original_weight=0.4):
    """Self-ensemble + TTA."""
    for m in models_list:
        m.eval()
    
    total = 0
    correct_top1, correct_top5 = 0, 0
    aug_weight = (1 - original_weight) / (n_augments - 1) if n_augments > 1 else 0
    
    for idx in tqdm(range(len(dataset)), desc="Ensemble+TTA"):
        landmarks, label = dataset.get_raw_landmarks(idx)
        
        final_logits = None
        
        # Para cada modelo del ensemble
        for model in models_list:
            weighted_logits = None
            
            # TTA para este modelo
            for i, transform in enumerate(TTA_TRANSFORMS[:n_augments]):
                lm_aug = transform(landmarks.copy())
                lm_aug = lm_aug.reshape(1, 64, -1)
                lm_tensor = torch.tensor(lm_aug, dtype=torch.float32).to(device)
                
                logits = model(lm_tensor)
                w = original_weight if i == 0 else aug_weight
                
                if weighted_logits is None:
                    weighted_logits = w * logits
                else:
                    weighted_logits += w * logits
            
            # Sumar al ensemble (peso igual para cada modelo)
            if final_logits is None:
                final_logits = weighted_logits / len(models_list)
            else:
                final_logits += weighted_logits / len(models_list)
        
        total += 1
        _, pred_top5 = final_logits.topk(5, 1, True, True)
        
        if pred_top5[0, 0] == label:
            correct_top1 += 1
        if label in pred_top5[0]:
            correct_top5 += 1
    
    return {
        'top1': 100 * correct_top1 / total,
        'top5': 100 * correct_top5 / total
    }


# Evaluar Self-Ensemble + TTA
final_val = evaluate_ensemble_with_tta(attention_models, val_dataset, n_augments=10, original_weight=0.4)
final_test = evaluate_ensemble_with_tta(attention_models, test_dataset, n_augments=10, original_weight=0.4)

print(f"\nSELF-ENSEMBLE + TTA:")
print(f"  Val Top-1:  {final_val['top1']:.1f}%")
print(f"  Test Top-1: {final_test['top1']:.1f}%")
print(f"  Test Top-5: {final_test['top5']:.1f}%")


COMBINACION FINAL: SELF-ENSEMBLE + TTA


Ensemble+TTA: 100%|██████████| 34/34 [00:02<00:00, 16.79it/s]


SELF-ENSEMBLE + TTA:
  Val Top-1:  67.9%
  Test Top-1: 55.9%
  Test Top-5: 85.3%


---
# RESUMEN FINAL
---

In [16]:
# ==========================================
# 16. RESUMEN FINAL
# ==========================================

print("\n" + "=" * 80)
print("RESUMEN FINAL - CAMINO AL 60%")
print("=" * 80)
print("")
print(f"{'Estrategia':<45} {'Val Top-1':<12} {'Test Top-1':<12} {'Test Top-5':<12}")
print("-" * 85)

# Baseline
print(f"{'GRU_Attention (baseline)':<45} {baseline_results['GRU_Attention']['val']['top1']:.1f}%{'':<7} {baseline_results['GRU_Attention']['test']['top1']:.1f}%{'':<7} {baseline_results['GRU_Attention']['test']['top5']:.1f}%")

# Weighted Ensemble
if best_weighted:
    print(f"{'Weighted Ensemble (' + best_weighted[1] + ')':<45} {best_weighted[2]['top1']:.1f}%{'':<7} {best_weighted[3]['top1']:.1f}%{'':<7} {best_weighted[3]['top5']:.1f}%")

# TTA Afinado
if best_tta:
    print(f"{'TTA Afinado (' + best_tta[2] + ')':<45} {best_tta[3]['top1']:.1f}%{'':<7} {best_tta[4]['top1']:.1f}%{'':<7} {best_tta[4]['top5']:.1f}%")

# Self-Ensemble
print(f"{'Self-Ensemble (3x GRU_Attention)':<45} {self_ensemble_val['top1']:.1f}%{'':<7} {self_ensemble_test['top1']:.1f}%{'':<7} {self_ensemble_test['top5']:.1f}%")

# Self-Ensemble + TTA
print(f"{'Self-Ensemble + TTA (LA BOMBA)':<45} {final_val['top1']:.1f}%{'':<7} {final_test['top1']:.1f}%{'':<7} {final_test['top5']:.1f}%")

print("")
print("=" * 80)
print(f"OBJETIVO DEL TUTOR: Top-1 60-70%")

best_result = max(final_test['top1'], self_ensemble_test['top1'], 
                  best_tta[4]['top1'] if best_tta else 0,
                  best_weighted[3]['top1'] if best_weighted else 0)

if best_result >= 60:
    print(f"\n*** OBJETIVO ALCANZADO: {best_result:.1f}% ***")
else:
    print(f"\nMejor resultado: {best_result:.1f}%")
    print(f"Faltan: {60 - best_result:.1f} puntos")


RESUMEN FINAL - CAMINO AL 60%

Estrategia                                    Val Top-1    Test Top-1   Test Top-5  
-------------------------------------------------------------------------------------
GRU_Attention (baseline)                      64.3%        58.8%        91.2%
Weighted Ensemble (70% Attention)             58.9%        61.8%        85.3%
TTA Afinado (5 augs, 30% original)            64.3%        61.8%        91.2%
Self-Ensemble (3x GRU_Attention)              67.9%        55.9%        85.3%
Self-Ensemble + TTA (LA BOMBA)                67.9%        55.9%        85.3%

OBJETIVO DEL TUTOR: Top-1 60-70%

*** OBJETIVO ALCANZADO: 61.8% ***


In [17]:
# ==========================================
# 17. GUARDAR RESULTADOS
# ==========================================

results = {
    'baseline': baseline_results,
    'weighted_ensemble': {
        'best_config': best_weighted[1] if best_weighted else None,
        'val': best_weighted[2] if best_weighted else None,
        'test': best_weighted[3] if best_weighted else None
    },
    'tta_afinado': {
        'best_config': best_tta[2] if best_tta else None,
        'val': best_tta[3] if best_tta else None,
        'test': best_tta[4] if best_tta else None
    },
    'self_ensemble': {
        'val': self_ensemble_val,
        'test': self_ensemble_test
    },
    'self_ensemble_tta': {
        'val': final_val,
        'test': final_test
    }
}

with open(OUTPUT_DIR / 'final_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Resultados guardados en: {OUTPUT_DIR}")

Resultados guardados en: /home/fransando/Dev/tfg-sign-language-recognition/05_experiments/final_push_60


## Conclusion

### Si llegaste al 60%:
- El Self-Ensemble de 3 modelos GRU_Attention + TTA es la combinacion ganadora
- La varianza entre seeds reduce el error

### Si aun no llegas:
- Entrenar mas modelos (5 seeds en vez de 3)
- Probar con mas datos de WLASL completo
- Knowledge distillation: entrenar un modelo estudiante del ensemble